<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%); padding: 40px 32px 32px 32px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: #ffffff; font-size: 2.0em; margin: 0 0 6px 0; font-family: 'Segoe UI', sans-serif; font-weight: 700;">
    🚗 Week 5, Lesson 9 -- Classification Models
  </h1>
  <h2 style="color: #a8d4f5; font-size: 1.2em; margin: 0 0 18px 0; font-family: 'Segoe UI', sans-serif; font-weight: 400;">
    Predicting Car Price Category from Specifications using Decision Trees
  </h2>
  <hr style="border: 1px solid rgba(255,255,255,0.25); margin: 16px 0;">
  <table style="color: #cce4ff; font-family: 'Segoe UI', sans-serif; font-size: 0.95em;">
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>📅 Week:</strong></td><td>5 (Lesson 9) -- Classification</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>⏱️ Duration:</strong></td><td>40 Minutes</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>🎯 Track:</strong></td><td>Data Science (Non-Petroleum)</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>👤 Audience:</strong></td><td>Data Scientists · Analysts · Researchers</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>🧑‍🏫 Instructor:</strong></td><td>Dr. Daniel Wamriew</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>✉️ Contact:</strong></td><td>wamriewdan@gmail.com</td>
    </tr>
  </table>
</div>

<div style="background: #e8f5e9; border-left: 5px solid #2ca87f; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### 💡 How to Use This Notebook

Work through this notebook **from top to bottom**, pressing **Shift + Enter** on each cell to run it.

- **Code cells** contain Python -- run them and read the output carefully.
- **Markdown cells** (white background, like this one) contain explanations -- read before running the next cell.
- **Student Activity** cells are marked with 🎯 -- complete these before moving on.
- **Homework** cells are marked with 📝 -- complete after the session.
- You need **`imports-85.data`** and **`imports-85.names`** in the same folder as this notebook.

</div>

---
## 📋 Table of Contents

1. [Why Classification?](#1-why-classification)
2. [Regression vs Classification](#2-regression-vs-classification)
3. [What is a Decision Tree?](#3-what-is-a-decision-tree)
4. [Load and Prepare the Data](#4-load-and-prepare-the-data)
5. [Build and Train the Model](#5-build-and-train-the-model)
6. [Evaluate the Model](#6-evaluate-the-model)
7. [Feature Importance](#7-feature-importance)
8. [Student Activity](#8-student-activity)
9. [Recap & Homework](#9-recap-homework)

---
## 1. Why Classification?

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

In Lesson 8 you predicted a **continuous number** (NPHI or SW) -- that is **regression**.

Many business problems require predicting a **category** instead:

| Business Question | Output | ML Task |
|-------------------|--------|----------|
| What will this car sell for? | A price in USD | Regression |
| Is this car budget, mid-range, or premium? | A label | Classification |
| Will this customer churn next month? | Yes / No | Classification |
| Which market segment does this product belong to? | A segment label | Classification |

In this lesson the model will look at a car's physical and mechanical specifications
and predict whether it falls in the **Low, Medium, or High** price category --
the kind of segmentation a data scientist at an import company needs every day.

</div>

---
## 2. Regression vs Classification

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

| | Regression | Classification |
|-|------------|----------------|
| **Predicts** | A number | A category / label |
| **Example output** | Price = 14 750 USD | Price category = "Medium" |
| **Loss function** | Mean Squared Error | Gini impurity / Cross-entropy |
| **Evaluation metric** | R², RMSE | Accuracy, Precision, Recall, F1 |
| **scikit-learn** | `LinearRegression()` | `DecisionTreeClassifier()` |

The 5-step ML workflow is **identical** -- only the model class and evaluation metrics change.

</div>

---
## 3. What is a Decision Tree?

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

A Decision Tree classifies data by asking a sequence of **if/else questions** about the features.
At each split it finds the question that best separates the classes.

**Example for car price:**
```
Is engine_size > 130?
    YES --> Is curb_weight > 3000?
                YES --> High price
                NO  --> Medium price
    NO  --> Is city_mpg > 30?
                YES --> Low price
                NO  --> Medium price
```

**Why Decision Trees are great for business data:**
- The learned rules are human-readable -- you can show them to a client
- They handle multiple classes naturally (Low / Medium / High)
- No need to scale or normalise features
- Feature importance reveals which car specs drive price most

</div>

---
## 4. Load and Prepare the Data

This is a real client dataset -- it has no column headers in the CSV file itself.
The column names are in `imports-85.names`, which we read first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

print("Libraries loaded.")

In [ ]:
# Load data -- no headers in the CSV
df = pd.read_csv("imports-85.data", header=None)

# Column names from imports-85.names
col_names = [
    "symboling", "normalized_losses", "make", "fuel_type", "aspiration",
    "num_of_doors", "body_style", "drive_wheels", "engine_location",
    "wheel_base", "length", "width", "height", "curb_weight",
    "engine_type", "num_of_cylinders", "engine_size", "fuel_system",
    "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm",
    "city_mpg", "highway_mpg", "price"
]
df.columns = col_names

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(4)

In [ ]:
# Replace '?' with NaN (the dataset uses '?' for missing values)
df = df.replace("?", np.nan)

# Convert numeric columns that were read as strings
numeric_cols = [
    "normalized_losses", "bore", "stroke", "horsepower",
    "peak_rpm", "price"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### Creating the Price Category Target

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

The original `price` column is a continuous number. We convert it into three market segments:

| Category | Price range | Business meaning |
|----------|-------------|------------------|
| **Low** | below USD 10 000 | Economy / entry-level |
| **Medium** | USD 10 000 -- 20 000 | Mid-range / mainstream |
| **High** | above USD 20 000 | Premium / luxury |

This is a deliberate business decision -- the thresholds would be agreed with the client.

</div>

In [ ]:
# Drop rows where price is missing (we cannot label them)
df = df.dropna(subset=["price"]).reset_index(drop=True)

# Create the price category target
def price_category(p):
    if p < 10000:
        return "Low"
    elif p < 20000:
        return "Medium"
    else:
        return "High"

df["price_category"] = df["price"].apply(price_category)

print("Price category distribution:")
print(df["price_category"].value_counts())
print(f"\nRows remaining: {len(df)}")

In [ ]:
# Select numeric features (no encoding needed for Decision Trees)
feature_cols = [
    "wheel_base", "length", "width", "height", "curb_weight",
    "engine_size", "compression_ratio", "horsepower",
    "city_mpg", "highway_mpg"
]

# Keep only rows with no missing values in the selected features
df_model = df[feature_cols + ["price_category"]].dropna().reset_index(drop=True)
print(f"Rows used for modelling: {len(df_model)}")

X = df_model[feature_cols]
y = df_model["price_category"]
class_order = ["Low", "Medium", "High"]

# 80/20 split, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training rows : {len(X_train)}")
print(f"Test rows     : {len(X_test)}")
print("\nClass balance in training set:")
print(y_train.value_counts())

---
## 5. Build and Train the Model

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

**`max_depth`** controls how many levels of questions the tree is allowed to ask.
A shallow tree (small `max_depth`) learns general patterns. A deep tree can memorise
the training data exactly -- this is **overfitting** and it performs poorly on new data.

</div>

In [ ]:
# Create and train the Decision Tree
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

print("Model trained!")
print(f"Tree depth (actual): {model.get_depth()}")
print(f"Number of leaves   : {model.get_n_leaves()}")

In [ ]:
# Visualise the decision tree
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(
    model,
    feature_names=feature_cols,
    class_names=class_order,
    filled=True,
    rounded=True,
    fontsize=7,
    ax=ax
)
ax.set_title("Decision Tree -- Car Price Category Classification", fontsize=11)
plt.tight_layout()
plt.show()
print("Each node: split condition | samples | class distribution | predicted class")

---
## 6. Evaluate the Model

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

| Metric | What it measures |
|--------|------------------|
| **Accuracy** | % of all predictions that were correct |
| **Precision** | Of all cars predicted as category X, how many actually were? |
| **Recall** | Of all cars that actually belong to category X, how many did the model find? |
| **F1 Score** | Harmonic mean of precision and recall |
| **Confusion Matrix** | Table of correct vs incorrect predictions per class |

</div>

In [ ]:
# Accuracy on training and test sets
y_pred     = model.predict(X_test)
train_acc  = accuracy_score(y_train, model.predict(X_train))
test_acc   = accuracy_score(y_test, y_pred)

print(f"Training accuracy : {train_acc:.3f}  ({train_acc*100:.1f}%)")
print(f"Test accuracy     : {test_acc:.3f}  ({test_acc*100:.1f}%)")

gap = train_acc - test_acc
if gap > 0.10:
    print(f"\nWarning: {gap:.2f} train-test gap -- consider reducing max_depth.")
else:
    print(f"\nTrain-test gap: {gap:.2f} -- model is generalising well.")

In [ ]:
# Full per-class report
print(classification_report(y_test, y_pred, target_names=class_order))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred, labels=class_order)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)

ax.set_xticks(list(range(len(class_order))))
ax.set_yticks(list(range(len(class_order))))
ax.set_xticklabels(class_order, fontsize=10)
ax.set_yticklabels(class_order, fontsize=10)
ax.set_xlabel("Predicted Category")
ax.set_ylabel("Actual Category")
ax.set_title("Confusion Matrix -- Car Price Category")

for i in range(len(class_order)):
    for j in range(len(class_order)):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=11, color=color)

plt.tight_layout()
plt.show()
print("Diagonal = correct predictions. Off-diagonal = errors.")

### Reading the Confusion Matrix

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

Each **row** is the actual price category. Each **column** is the model's prediction.

- **Diagonal cells** -- correct predictions. You want these numbers to be high.
- **Off-diagonal cells** -- mistakes. If the model confuses Medium with High, that cell will be non-zero.
- The most common error in price classification is at the **boundaries** -- a car priced at USD 9 800
  and one at USD 10 200 have almost identical specs, so the model may classify both the same way.
  This is expected and acceptable.

</div>

---
## 7. Feature Importance

Which car specifications did the model rely on most when deciding the price category?

In [ ]:
importances  = model.feature_importances_
sorted_idx   = list(np.argsort(importances)[::-1])
sorted_names = [feature_cols[i] for i in sorted_idx]
sorted_scores= [importances[i] for i in sorted_idx]

print("Feature Importance (most to least influential):")
for name, score in zip(sorted_names, sorted_scores):
    bar = '#' * int(score * 40)
    print(f"  {name:20s}  {score:.4f}  {bar}")

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#1a3a5c", "#2d6a9f", "#2ca87f", "#e07b39", "#8e44ad",
          "#c0392b", "#27ae60", "#d35400", "#2980b9", "#6c3483"]
ax.barh(sorted_names[::-1], sorted_scores[::-1], color=colors, edgecolor="white")
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importance for Car Price Classification")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

---
## 8. Student Activity

<div style="background: #fff8e1; border-left: 5px solid #e07b39; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### 🎯 Tune the Tree and Interpret the Results

Retrain the model with `max_depth=2` and then with `max_depth=6`. For each:

1. Print train and test accuracy
2. Print the classification report
3. Fill in the table below

| max_depth | Train Accuracy | Test Accuracy | Best class | Worst class |
|-----------|---------------|---------------|------------|-------------|
| 2 | | | | |
| 4 (lesson model) | | | | |
| 6 | | | | |

**Questions to answer:**
- Which price category is hardest to classify? Why might that be?
- At `max_depth=6`, is training accuracy significantly higher than test accuracy?
  What is the name of this problem, and how can you fix it?
- Based on feature importance, which **one** car specification is the strongest
  predictor of price category? Does this make sense to you?

</div>

In [ ]:
# 🎯 Your code here

# max_depth = 2


# max_depth = 6



*Write your answers here:*

- Hardest category to classify and why:
- max_depth=6 train vs test accuracy observation:
- Strongest single predictor and why it makes sense:


---
## 9. Recap & Homework

### 📝 What We Covered in Lesson 9

| Concept | Key Point |
|---------|----------|
| Classification | Predicts a label/category instead of a number |
| Decision Tree | Learns a sequence of if/else rules from data; outputs are human-readable |
| Price binning | Converting a continuous variable into categories is a design decision |
| `max_depth` | Controls overfitting -- too deep memorises; too shallow underfits |
| `stratify=y` | Preserves class balance in train/test split |
| Accuracy | Overall % correct |
| Precision / Recall / F1 | Per-class metrics that reveal imbalance issues |
| Confusion matrix | Shows which categories are confused with each other |
| Feature importance | Reveals which specifications drive the prediction most |

### 📝 Homework

<div style="background: #fff8e1; border-left: 5px solid #f39c12; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0;">

**Task 1 -- Binary Classification (All Students)**

Create a binary target: `premium` = 1 if `price_category == 'High'`, else 0.
Train a Decision Tree (`max_depth=3`) to predict this flag.
Report accuracy, precision, recall, and F1 for the positive class (1 = premium).
Which features are most important for identifying premium cars?

</div>

<div style="background: #fff8e1; border-left: 5px solid #f39c12; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0;">

**Task 2 -- Categorical Feature (All Students)**

The model in this lesson used only numeric features. The `body_style` column
(sedan, hatchback, wagon, etc.) was ignored because it is a string.

Use `pd.get_dummies(df_model, columns=["body_style"])` to encode it numerically,
then add the new dummy columns to your feature set and retrain the model.
Does test accuracy improve?

</div>

<div style="background: #fff8e1; border-left: 5px solid #f39c12; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0;">

**Task 3 -- Reframe the Labels (Data Science Track)**

Redefine the price categories using custom thresholds that make sense
for a different market (e.g., Budget below USD 7 000, Mid USD 7 000 -- 15 000, Premium above).
Retrain the model and compare class distribution and accuracy with the original split.
Write two sentences explaining how the threshold choice affects model performance
and why this is a business decision, not just a technical one.

</div>

In [ ]:
# 📝 Task 1 -- Binary: premium vs non-premium
# Your code here


In [ ]:
# 📝 Task 2 -- Add body_style with get_dummies
# Your code here


In [ ]:
# 📝 Task 3 -- Custom price thresholds
# Your code here


---
<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%); padding: 24px 28px; border-radius: 10px; margin-top: 16px; text-align: center;">
  <h3 style="color: #ffffff; margin: 0 0 10px 0;">Lesson 9 Complete!</h3>
  <p style="color: #a8d4f5; margin: 0 0 10px 0;">
    You have built a classification model on a real business dataset, handled messy real-world data,
    created meaningful labels from a continuous variable, and evaluated the model like a practising data scientist.
  </p>
  <p style="color: #d0eaff; font-size: 0.9em; margin: 0;">
    Next: <strong>Lesson 10 -- Model Improvement: Cross-Validation &amp; Random Forests</strong>
  </p>
</div>